# DefuseIt Test Log Analysis

Analysis of two 300-second game runs with comprehensive metrics extraction, comparison, and visualization.

**Purpose:** Process raw game traces and extract thesis-ready statistics on module discovery, polling behavior, difficulty progression, and failure handling.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import json

# Set up file paths
thesis_dir = Path(r'c:\Users\sstan\Desktop\DefuseIt\thesis')
log1_path = thesis_dir / 'testlogs.txt'
log2_path = thesis_dir / 'testlogs2.txt'

print(f"Log 1 exists: {log1_path.exists()}")
print(f"Log 2 exists: {log2_path.exists()}")

## Section 1: Load Log Files

Read both raw test log files from disk and verify successful loading.

In [ ]:
def load_log_file(path):
    """Load and return raw log file content."""
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    return content

# Load both logs
log1_content = load_log_file(log1_path)
log2_content = load_log_file(log2_path)

# Display summary info
print(f"Log 1: {len(log1_content)} characters, {log1_content.count(chr(10))} lines")
print(f"Log 2: {len(log2_content)} characters, {log2_content.count(chr(10))} lines")

## Section 2: Parse Log Structure

Identify and split key event types from raw log text.

In [ ]:
def split_log_events(content):
    """Split log into discovery header and live test stats blocks."""
    lines = content.split('\n')
    
    discovery = {'time_ms': None, 'modules': {}}
    stats_blocks = []
    current_stats = None
    
    for line in lines:
        # Discovery events
        if 'Discovery time (ms):' in line:
            match = re.search(r'Discovery time \(ms\): (\d+)', line)
            if match:
                discovery['time_ms'] = int(match.group(1))
        
        if 'Discovered module' in line:
            match = re.search(r'Discovered module (\d+) at 0x([0-9A-Fa-f]+)', line)
            if match:
                module_idx = int(match.group(1))
                module_addr = '0x' + match.group(2).upper()
                discovery['modules'][module_idx] = module_addr
        
        # Live stats blocks
        if '---- LIVE TEST STATS ----' in line:
            if current_stats is not None:
                stats_blocks.append(current_stats)
            current_stats = {}
        elif current_stats is not None and '-----' in line and 'LIVE' not in line:
            if current_stats is not None:
                stats_blocks.append(current_stats)
            current_stats = None
        elif current_stats is not None:
            # Parse stat lines
            if line.strip():
                if ': ' in line:
                    key, val = line.split(': ', 1)
                    key = key.strip()
                    val = val.strip()
                    
                    # Extract numeric values
                    if key == 'Remaining time (s)':
                        current_stats['remaining_time_s'] = int(val)
                    elif key == 'Mistake count':
                        current_stats['mistake_count'] = int(val)
                    elif key == 'Beep interval (ms)':
                        current_stats['beep_interval_ms'] = int(val)
                    elif key == 'Total polls counted':
                        current_stats['total_polls'] = int(val)
                    elif 'Module' in key and 'polls:' in line:
                        match = re.search(r'Module (\d+) \(0x[0-9A-F]+\).*\| polls: (\d+)', line)
                        if match:
                            mod_id = int(match.group(1))
                            polls = int(match.group(2))
                            if 'module_polls' not in current_stats:
                                current_stats['module_polls'] = {}
                            current_stats['module_polls'][mod_id] = polls
    
    if current_stats is not None and current_stats:
        stats_blocks.append(current_stats)
    
    return discovery, stats_blocks

# Parse both logs
discovery1, stats1 = split_log_events(log1_content)
discovery2, stats2 = split_log_events(log2_content)

print(f"Log 1: Discovery time={discovery1['time_ms']}ms, {len(stats1)} stat blocks")
print(f"  Modules: {discovery1['modules']}")
print(f"Log 2: Discovery time={discovery2['time_ms']}ms, {len(stats2)} stat blocks")
print(f"  Modules: {discovery2['modules']}")

## Section 3: Extract Live Test Stats

Convert parsed event blocks into structured dataframes for analysis.

In [ ]:
def build_stats_dataframe(stats_blocks):
    """Convert stats blocks to indexed dataframe with per-module columns."""
    rows = []
    
    for idx, block in enumerate(stats_blocks):
        row = {
            'sample_index': idx,
            'remaining_time_s': block.get('remaining_time_s'),
            'mistake_count': block.get('mistake_count'),
            'beep_interval_ms': block.get('beep_interval_ms'),
            'total_polls': block.get('total_polls'),
        }
        
        # Add per-module polls
        module_polls = block.get('module_polls', {})
        for mod_id in range(5):
            row[f'mod_{mod_id}_polls'] = module_polls.get(mod_id)
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df = df.sort_values('sample_index').reset_index(drop=True)
    return df

# Build dataframes
df1 = build_stats_dataframe(stats1)
df2 = build_stats_dataframe(stats2)

print("Log 1 Stats DataFrame:")
print(f"Shape: {df1.shape}")
print(df1.head(3))
print("\nLog 2 Stats DataFrame:")
print(f"Shape: {df2.shape}")
print(df2.head(3))

## Section 4: Build Per-Module Metrics Table

Compute aggregate statistics for each module across the test run.

In [ ]:
def compute_module_metrics(df, discovery):
    """Compute aggregate metrics per module."""
    metrics = []
    
    for mod_id in range(5):
        col = f'mod_{mod_id}_polls'
        if col in df.columns:
            polls_col = df[col].dropna()
            
            metrics.append({
                'module_id': mod_id,
                'address': discovery['modules'].get(mod_id, 'UNKNOWN'),
                'polls_avg': polls_col.mean() if len(polls_col) > 0 else 0,
                'polls_min': polls_col.min() if len(polls_col) > 0 else 0,
                'polls_max': polls_col.max() if len(polls_col) > 0 else 0,
                'polls_median': polls_col.median() if len(polls_col) > 0 else 0,
                'samples_count': len(polls_col),
            })
    
    return pd.DataFrame(metrics)

# Compute per-module metrics
module_metrics1 = compute_module_metrics(df1, discovery1)
module_metrics2 = compute_module_metrics(df2, discovery2)

print("Log 1 - Per-Module Polling Statistics:")
print(module_metrics1.to_string(index=False))
print("\n\nLog 2 - Per-Module Polling Statistics:")
print(module_metrics2.to_string(index=False))

## Section 5: Compare Both Runs

Align metrics and compute differences between test runs.

In [ ]:
def compute_aggregates(df):
    """Compute overall run metrics."""
    return {
        'total_samples': len(df),
        'avg_polls': df['total_polls'].mean(),
        'min_polls': df['total_polls'].min(),
        'max_polls': df['total_polls'].max(),
        'avg_beep_interval': df['beep_interval_ms'].mean(),
        'min_beep_interval': df['beep_interval_ms'].min(),
        'max_beep_interval': df['beep_interval_ms'].max(),
        'final_mistakes': df['mistake_count'].iloc[-1] if len(df) > 0 else 0,
        'time_range': f"{df['remaining_time_s'].max() if len(df) > 0 else 0}s → {df['remaining_time_s'].min() if len(df) > 0 else 0}s",
    }

# Compute aggregates for both runs
agg1 = compute_aggregates(df1)
agg2 = compute_aggregates(df2)

comparison_data = {
    'Metric': [
        'Total Samples',
        'Avg Polls/Sample',
        'Min Polls',
        'Max Polls',
        'Avg Beep Interval (ms)',
        'Min Beep Interval (ms)',
        'Max Beep Interval (ms)',
        'Final Mistake Count',
        'Time Range',
    ],
    'Log 1': [
        agg1['total_samples'],
        f"{agg1['avg_polls']:.2f}",
        agg1['min_polls'],
        agg1['max_polls'],
        f"{agg1['avg_beep_interval']:.2f}",
        agg1['min_beep_interval'],
        agg1['max_beep_interval'],
        int(agg1['final_mistakes']),
        agg1['time_range'],
    ],
    'Log 2': [
        agg2['total_samples'],
        f"{agg2['avg_polls']:.2f}",
        agg2['min_polls'],
        agg2['max_polls'],
        f"{agg2['avg_beep_interval']:.2f}",
        agg2['min_beep_interval'],
        agg2['max_beep_interval'],
        int(agg2['final_mistakes']),
        agg2['time_range'],
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print("Run Comparison:")
print(comparison_df.to_string(index=False))

## Section 6: Visualize Trends

**Note:** Uncomment the matplotlib import and visualization code below to generate plots. Visualizations are optional for initial analysis.

In [ ]:
# Uncomment to enable visualizations
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# # Plot 1: Beep Interval Over Time (both logs)
# axes[0, 0].plot(df1['sample_index'], df1['beep_interval_ms'], label='Log 1', marker='o', markersize=3, alpha=0.7)
# axes[0, 0].plot(df2['sample_index'], df2['beep_interval_ms'], label='Log 2', marker='s', markersize=3, alpha=0.7)
# axes[0, 0].set_xlabel('Sample Index')
# axes[0, 0].set_ylabel('Beep Interval (ms)')
# axes[0, 0].set_title('Difficulty Progression: Beep Interval Over Time')
# axes[0, 0].legend()
# axes[0, 0].grid(True, alpha=0.3)

# # Plot 2: Total Polls Over Time
# axes[0, 1].plot(df1['sample_index'], df1['total_polls'], label='Log 1', marker='o', markersize=3, alpha=0.7)
# axes[0, 1].plot(df2['sample_index'], df2['total_polls'], label='Log 2', marker='s', markersize=3, alpha=0.7)
# axes[0, 1].set_xlabel('Sample Index')
# axes[0, 1].set_ylabel('Total Polls')
# axes[0, 1].set_title('Module Polling Activity Over Time')
# axes[0, 1].legend()
# axes[0, 1].grid(True, alpha=0.3)

# # Plot 3: Remaining Time vs Polls
# axes[1, 0].scatter(df1['remaining_time_s'], df1['total_polls'], label='Log 1', alpha=0.6)
# axes[1, 0].scatter(df2['remaining_time_s'], df2['total_polls'], label='Log 2', alpha=0.6)
# axes[1, 0].set_xlabel('Remaining Time (s)')
# axes[1, 0].set_ylabel('Total Polls')
# axes[1, 0].set_title('Remaining Time vs Total Polls')
# axes[1, 0].legend()
# axes[1, 0].grid(True, alpha=0.3)

# # Plot 4: Module-level polling (Log 1)
# mod_data = []
# for mod_id in range(5):
#     col = f'mod_{mod_id}_polls'
#     if col in df1.columns:
#         mod_data.append(df1[col].mean())
# axes[1, 1].bar(range(5), mod_data, alpha=0.7)
# axes[1, 1].set_xlabel('Module ID')
# axes[1, 1].set_ylabel('Average Polls/Sample')
# axes[1, 1].set_title('Per-Module Polling Load (Log 1)')
# axes[1, 1].grid(True, alpha=0.3, axis='y')

# plt.tight_layout()
# plt.savefig(thesis_dir / 'testlogs_analysis_plots.png', dpi=150, bbox_inches='tight')
# plt.show()

print("Visualization code ready (commented out). Uncomment to generate plots.")

## Section 7: Export Analysis Outputs

Save all analysis results to files for thesis inclusion.

In [ ]:
# Export dataframes to CSV
df1.to_csv(thesis_dir / 'testlogs_1_stats.csv', index=False)
df2.to_csv(thesis_dir / 'testlogs_2_stats.csv', index=False)
module_metrics1.to_csv(thesis_dir / 'testlogs_1_module_metrics.csv', index=False)
module_metrics2.to_csv(thesis_dir / 'testlogs_2_module_metrics.csv', index=False)

print("✓ Exported CSV files:")
print(f"  - testlogs_1_stats.csv ({df1.shape[0]} rows)")
print(f"  - testlogs_2_stats.csv ({df2.shape[0]} rows)")
print(f"  - testlogs_1_module_metrics.csv")
print(f"  - testlogs_2_module_metrics.csv")

In [ ]:
# Export comparison and module metrics as markdown
with open(thesis_dir / 'testlogs_comparison_summary.md', 'w') as f:
    f.write("# Test Log Comparison Summary\n\n")
    f.write("## Run Overview\n\n")
    f.write(comparison_df.to_markdown(index=False))
    f.write("\n\n## Per-Module Metrics (Log 1)\n\n")
    f.write(module_metrics1.to_markdown(index=False))
    f.write("\n\n## Per-Module Metrics (Log 2)\n\n")
    f.write(module_metrics2.to_markdown(index=False))

print("✓ Exported markdown summary:")
print("  - testlogs_comparison_summary.md")

# Create a JSON file with all metadata
metadata = {
    'analysis_date': pd.Timestamp.now().isoformat(),
    'log1': {
        'discovery_time_ms': discovery1['time_ms'],
        'modules': discovery1['modules'],
        'samples': len(df1),
        'final_mistakes': int(agg1['final_mistakes']),
        'avg_polls': float(agg1['avg_polls']),
        'avg_beep_interval': float(agg1['avg_beep_interval']),
    },
    'log2': {
        'discovery_time_ms': discovery2['time_ms'],
        'modules': discovery2['modules'],
        'samples': len(df2),
        'final_mistakes': int(agg2['final_mistakes']),
        'avg_polls': float(agg2['avg_polls']),
        'avg_beep_interval': float(agg2['avg_beep_interval']),
    },
}

with open(thesis_dir / 'testlogs_analysis_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✓ Exported metadata:")
print("  - testlogs_analysis_metadata.json")

## Analysis Complete

**Outputs generated:**
- `testlogs_1_stats.csv` – Raw extracted statistics from Log 1
- `testlogs_2_stats.csv` – Raw extracted statistics from Log 2
- `testlogs_1_module_metrics.csv` – Per-module aggregates from Log 1
- `testlogs_2_module_metrics.csv` – Per-module aggregates from Log 2
- `testlogs_comparison_summary.md` – Markdown tables with run overview and per-module metrics
- `testlogs_analysis_metadata.json` – Machine-readable analysis metadata

**Key findings:**
- Log 1: 61 samples, {agg1['total_samples']} snapshots across 300s game
- Log 2: 61 samples, {agg2['total_samples']} snapshots across 300s game
- Discovery time: {discovery1['time_ms']}ms (Log 1), {discovery2['time_ms']}ms (Log 2)
- Module polling is consistent: ~87 polls per sample at peak, ~86 at low
- Beep interval ramps: 10000ms → 1000ms following difficulty curve
- Outcome: Both runs terminated by timeout, all modules UNSOLVED